# Maximum Likelihood Estimation (MLE) — From First Principles with Worked Examples

_Generated: 2025-10-22T16:59:01.790277Z_

**Objective.** Build MLE from first principles and solve concrete estimation problems analytically and numerically. We derive estimators, visualize likelihoods, compute asymptotic standard errors via the observed Fisher information, and validate via a small bootstrap.

**Contents**
1. Concept recap: likelihood vs. probability; log-likelihood; score and information.
2. Analytic MLEs: Bernoulli coin bias, Poisson rate, Exponential rate, Gaussian mean/variance.
3. Numerical MLE: Cauchy location parameter (no finite moments) via direct log-likelihood maximization.
4. Inference: Wald intervals from observed information and a nonparametric bootstrap.


## 0. Environment & Dependencies

We use `numpy` and `matplotlib`. `scipy` is optional and used only for cross-checking (commented by default).

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

# Optional:
# !pip -q install scipy

## 1. Imports, Reproducibility, and Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

def plot_ll(theta_grid, ll_values, theta_hat, true_theta=None, xlabel=r"$\theta$", title="Log-likelihood"):
    fig = plt.figure(figsize=(6,4))
    plt.plot(theta_grid, ll_values)
    if theta_hat is not None:
        plt.axvline(theta_hat, linestyle="--")
    if true_theta is not None:
        plt.axvline(true_theta, linestyle=":")
    plt.title(title)
    plt.xlabel(xlabel); plt.ylabel("log L(θ | x)")
    plt.tight_layout()
    plt.show()

def wald_ci(theta_hat, I_obs, alpha=0.05):
    # theta_hat: scalar; I_obs: observed Fisher information at theta_hat
    from math import sqrt
    se = 1.0/np.sqrt(I_obs + 1e-12)
    z = 1.959963984540054  # approx for 0.975
    return theta_hat - z*se, theta_hat + z*se, se

## 2. Concept Recap

**Likelihood** of parameter $\theta$ given data $x_{1:n}$ under model $p(x \mid \theta)$ is $\mathcal{L}(\theta;x) = \prod_{i=1}^n p(x_i \mid \theta)$. The **log-likelihood** $\ell(\theta) = \sum_i \log p(x_i\mid\theta)$ is easier to maximize. The **score** is $\partial \ell/\partial \theta$ and the **observed information** is $\mathcal{I}(\theta) = -\partial^2 \ell/\partial\theta^2$. Under regularity, $\hat\theta_{MLE}\overset{approx}{\sim} \mathcal{N}(\theta, 1/\mathcal{I}(\hat\theta))$.

## 3. Analytic MLEs — Bernoulli (Coin Bias)

Model: i.i.d. $X_i\sim\text{Bernoulli}(p)$. Log-likelihood: $\ell(p)=\sum_i [x_i\log p + (1-x_i)\log(1-p)]$. Setting derivative to zero yields $\hat p = \bar x$. Observed information: $\mathcal{I}(p)=n\big(\frac{1}{p(1-p)}\big)\bar x(1-\bar x)$ evaluated at $\hat p$ simplifies to $n/(\hat p(1-\hat p))$.

In [ ]:
# Simulate data
n = 200
p_true = 0.63
x = rng.binomial(1, p_true, size=n)
p_hat = x.mean()

# Observed information (exact for Bernoulli at MLE): I = n / [p(1-p)]
I_obs = n / (p_hat*(1-p_hat) + 1e-12)
ci_lo, ci_hi, se = wald_ci(p_hat, I_obs)

print(f"Bernoulli: n={n}, true p={p_true:.3f}, MLE p̂={p_hat:.3f}, SE≈{se:.4f}, 95% CI=({ci_lo:.3f}, {ci_hi:.3f})")

# Visualize log-likelihood
grid = np.linspace(0.01, 0.99, 300)
ll = x.sum()*np.log(grid) + (n - x.sum())*np.log(1-grid)
plot_ll(grid, ll, p_hat, true_theta=p_true, xlabel=r"$p$", title="Bernoulli log-likelihood")

## 4. Analytic MLE — Poisson Rate

Model: $X_i\sim\text{Poisson}(\lambda)$. Log-likelihood (up to const): $\ell(\lambda)=\sum_i [x_i\log\lambda - \lambda]$. Derivative $\Rightarrow \hat\lambda = \bar x$. Observed information: $\mathcal{I}(\lambda)=n/\lambda$ (evaluate at $\hat\lambda$).

In [ ]:
m = 150
lam_true = 2.3
y = rng.poisson(lam_true, size=m)
lam_hat = y.mean()
I_obs = m / (lam_hat + 1e-12)
ci_lo, ci_hi, se = wald_ci(lam_hat, I_obs)
print(f"Poisson: m={m}, true λ={lam_true:.3f}, MLE λ̂={lam_hat:.3f}, SE≈{se:.4f}, 95% CI=({ci_lo:.3f}, {ci_hi:.3f})")

grid = np.linspace(0.1, 6.0, 300)
ll = y.sum()*np.log(grid) - m*grid
plot_ll(grid, ll, lam_hat, true_theta=lam_true, xlabel=r"$\lambda$", title="Poisson log-likelihood")

## 5. Analytic MLE — Exponential Rate

Model: $X_i\sim\text{Exponential}(\lambda)$ with density $\lambda e^{-\lambda x}$ on $x\ge0$. Log-likelihood: $\ell(\lambda)=n\log\lambda - \lambda\sum x_i$. Derivative $\Rightarrow \hat\lambda = 1/\bar x$. Observed information: $\mathcal{I}(\lambda)=n/\lambda^2$. For the **scale** $\theta=1/\lambda$, the MLE is the sample mean.

In [ ]:
k = 300
lam_true = 1.8
z = rng.exponential(1/lam_true, size=k)  # exponential with mean 1/λ
lam_hat = 1.0 / z.mean()
I_obs = k / (lam_hat**2 + 1e-12)
ci_lo, ci_hi, se = wald_ci(lam_hat, I_obs)
print(f"Exponential: k={k}, true λ={lam_true:.3f}, MLE λ̂={lam_hat:.3f}, SE≈{se:.4f}, 95% CI=({ci_lo:.3f}, {ci_hi:.3f})")

grid = np.linspace(0.1, 5.0, 300)
ll = k*np.log(grid) - grid * z.sum()
plot_ll(grid, ll, lam_hat, true_theta=lam_true, xlabel=r"$\lambda$", title="Exponential log-likelihood")

## 6. Analytic MLE — Gaussian Mean and Variance

Model: $X_i\sim\mathcal{N}(\mu,\sigma^2)$ i.i.d.

- If $\sigma^2$ known, $\hat\mu=\bar x$.
- If both unknown, $\hat\mu=\bar x$ and $\hat\sigma^2 = \frac{1}{n}\sum (x_i-\bar x)^2$ (note: **ML** uses $1/n$, not $1/(n-1)$).

In [ ]:
n = 250
mu_true, sigma_true = 2.0, 1.5
x = rng.normal(mu_true, sigma_true, size=n)
mu_hat = x.mean()
sigma2_hat = np.mean((x - mu_hat)**2)  # ML variance

# Observed information for μ with σ known is n/σ^2. With σ unknown, asymptotically similar using σ̂.
I_obs_mu = n / (sigma2_hat + 1e-12)
ci_lo, ci_hi, se = wald_ci(mu_hat, I_obs_mu)
print(f"Gaussian: n={n}, true μ={mu_true:.3f}, MLE μ̂={mu_hat:.3f}, SE≈{se:.4f}, 95% CI=({ci_lo:.3f}, {ci_hi:.3f})")
print(f"Gaussian: MLE σ̂^2={sigma2_hat:.3f} (compare true σ^2={sigma_true**2:.3f})")

grid = np.linspace(mu_true-3.0, mu_true+3.0, 300)
ll = -0.5*n*np.log(2*np.pi*sigma2_hat) - 0.5*np.sum((x[:,None] - grid[None,:])**2, axis=0)/sigma2_hat
plot_ll(grid, ll, mu_hat, true_theta=mu_true, xlabel=r"$\mu$", title="Gaussian log-likelihood (profile in μ)")

## 7. Numerical MLE — Cauchy Location (Heavy Tails)

For a Cauchy($\theta$, $\gamma$) with known scale $\gamma$, the MLE for **location** $\theta$ has no closed form. We maximize $\ell(\theta)=\sum_i -\log(\gamma^2 + (x_i-\theta)^2)$ (up to const). We perform a robust 1-D search to illustrate numerical MLE.

In [ ]:
# Generate Cauchy samples (using tangent formula). We'll fix gamma=1.0 (scale).
n = 200
theta_true, gamma = 0.5, 1.0
u = rng.uniform(0, 1, size=n)
x = theta_true + gamma * np.tan(np.pi*(u - 0.5))

def cauchy_loglik(theta, x, gamma=1.0):
    return -np.sum(np.log(gamma**2 + (x - theta)**2))

# Coarse-to-fine grid search
g1 = np.linspace(np.percentile(x, 5)-2, np.percentile(x,95)+2, 600)
ll1 = np.array([cauchy_loglik(t, x, gamma) for t in g1])
t1 = g1[np.argmax(ll1)]
# refine
g2 = np.linspace(t1-0.5, t1+0.5, 600)
ll2 = np.array([cauchy_loglik(t, x, gamma) for t in g2])
theta_hat = g2[np.argmax(ll2)]

print(f"Cauchy location MLE (grid search): true θ={theta_true:.3f}, θ̂={theta_hat:.3f}")

plot_ll(g1, ll1, t1, true_theta=theta_true, xlabel=r"$\theta$", title="Cauchy log-likelihood (coarse)")
plot_ll(g2, ll2, theta_hat, true_theta=theta_true, xlabel=r"$\theta$", title="Cauchy log-likelihood (refined)")

### 7.1 Wald Interval via Observed Information (Numeric Hessian)

We approximate the observed information by the negative second derivative of the log-likelihood at $\hat\theta$ using a small step $h$. This gives an approximate standard error and Wald CI.

In [ ]:
def numeric_second_derivative(f, theta, h=1e-3):
    return (f(theta+h) - 2*f(theta) + f(theta-h)) / (h*h)

f = lambda th: cauchy_loglik(th, x, gamma)
I_obs_num = -numeric_second_derivative(f, theta_hat, h=1e-3)
ci_lo, ci_hi, se = wald_ci(theta_hat, I_obs_num)
print(f"Cauchy: θ̂={theta_hat:.3f}, observed I≈{I_obs_num:.3f}, SE≈{se:.4f}, 95% CI=({ci_lo:.3f}, {ci_hi:.3f})")

## 8. Bootstrap Confidence Interval (Nonparametric)

We resample the data with replacement, recompute the MLE for each bootstrap sample, and form a percentile CI. This is distribution-free (conditional on the data) and often more robust for small n or non-regular problems.

In [ ]:
def cauchy_mle_grid(x, gamma=1.0):
    g = np.linspace(np.percentile(x, 5)-2, np.percentile(x,95)+2, 500)
    ll = np.array([cauchy_loglik(t, x, gamma) for t in g])
    t = g[np.argmax(ll)]
    # quick refine
    g2 = np.linspace(t-0.5, t+0.5, 400)
    ll2 = np.array([cauchy_loglik(t, x, gamma) for t in g2])
    return g2[np.argmax(ll2)]

B = 200
thetas = []
for b in range(B):
    xb = rng.choice(x, size=len(x), replace=True)
    thetas.append(cauchy_mle_grid(xb, gamma))
thetas = np.array(thetas)
lo, hi = np.percentile(thetas, [2.5, 97.5])
print(f"Bootstrap percentile 95% CI for θ: ({lo:.3f}, {hi:.3f})")

## 9. Save Artifacts & Download

We persist simulation draws, MLEs, grids, and CI summaries for all examples. Use the helper below to download a ZIP in Colab.

In [ ]:
import os, json
os.makedirs("artifacts", exist_ok=True)

# Save a minimal JSON summary (you can extend as desired)
summary = {
    "bernoulli": {"n": int(n), "p_true": float(p_true), "p_hat": float(p_hat)},
    "poisson": {"m": int(m), "lambda_true": float(lam_true), "lambda_hat": float(lam_hat)},
    "exponential": {"k": int(k), "lambda_true": float(lam_true), "lambda_hat": float(lam_hat)},
    "gaussian": {"n": int(n), "mu_true": float(mu_true), "mu_hat": float(mu_hat), "sigma2_hat": float(sigma2_hat)},
    "cauchy": {"theta_true": float(theta_true), "theta_hat": float(theta_hat)}
}
with open("artifacts/mle_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Save bootstrap samples
np.savez("artifacts/cauchy_bootstrap_thetas.npz", thetas=thetas)

print("Saved artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 10. Appendix: Extensions and Exercises

- Derive MLEs for **Binomial** $n$ known; **Gamma** with fixed shape; **Negative Binomial**; **Beta** with one parameter fixed.
- Compare Wald intervals with **likelihood ratio** intervals (invert LR tests) and **score** intervals.
- Implement a **generic 1-D optimizer** (Newton–Raphson) using the score and observed information; compare convergence.
- For multivariate parameters, add a **numerical Hessian** and compute the covariance matrix $\hat{\mathcal{I}}^{-1}$.
- Explore small-sample behavior via simulation; contrast ML variance vs unbiased variance in the Gaussian case.